## <font size=5> <strong> Quick Look at the Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Loading data
df = pd.read_csv('Titanic-Dataset.csv')

# See what we're working with in dataset
print("First 5 passengers:")
print(df.head())



First 5 passengers:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500

In [3]:
print("\nBasic info:")
print(df.info())


Basic info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


## <font size=5> <strong> Pattern Check

In [4]:
# Simple question: Did women survive more than men?
women_survived = df[df['Sex'] == 'female']['Survived'].mean() * 100
men_survived = df[df['Sex'] == 'male']['Survived'].mean() * 100

print(f"Women who survived: {women_survived:.1f}%")
print(f"Men who survived: {men_survived:.1f}%")


Women who survived: 74.2%
Men who survived: 18.9%


## <font size=5> <strong>Fix Missing Info 

In [5]:
# Create a clean copy
clean_df = df.copy()

In [6]:
# Fill missing ages with average age
clean_df['Age'].fillna(clean_df['Age'].median(), inplace=True)

C:\Users\CVNS\AppData\Local\Temp\ipykernel_3712\3283843991.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  clean_df['Age'].fillna(clean_df['Age'].median(), inplace=True)


In [7]:
# Fill missing embark port with most common port
clean_df['Embarked'].fillna('S', inplace=True)


C:\Users\CVNS\AppData\Local\Temp\ipykernel_3712\3698170536.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  clean_df['Embarked'].fillna('S', inplace=True)


In [8]:
# Fill missing fare with average fare
clean_df['Fare'].fillna(clean_df['Fare'].median(), inplace=True)

C:\Users\CVNS\AppData\Local\Temp\ipykernel_3712\2352145034.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  clean_df['Fare'].fillna(clean_df['Fare'].median(), inplace=True)


## <font size=5> <strong>Create New Helpful Info

In [9]:
# Family size (were they alone or with family?)
clean_df['FamilySize'] = clean_df['SibSp'] + clean_df['Parch'] + 1

# Did they travel alone?
clean_df['Alone'] = (clean_df['FamilySize'] == 1).astype(int)

# Convert text to numbers (computers like numbers better)
clean_df['Sex_Number'] = (clean_df['Sex'] == 'female').astype(int)

 ## <font size=5> <strong>Pick What Info to Use

In [10]:
# Choose the best clues to predict survival
features = ['Pclass', 'Sex_Number', 'Age', 'FamilySize', 'Fare', 'Alone']
X = clean_df[features]  # The clues
y = clean_df['Survived']  # The answer we want to predict

## <font size=5> <strong>Train My Predictor

In [11]:
# Split data: 80% for learning, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Create and train the model (like teaching a student)
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Check how accurate it is
accuracy = model.score(X_test, y_test)
print(f"My predictor is {accuracy*100:.1f}% accurate!")

My predictor is 79.9% accurate!


## <font size=5> <strong>See What Mattered Most

In [12]:
# What clues were most important?
importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nWhat mattered most for survival:")
print(importance)


What mattered most for survival:
      Feature  Importance
4        Fare    0.312403
2         Age    0.280414
1  Sex_Number    0.230824
0      Pclass    0.095255
3  FamilySize    0.068388
5       Alone    0.012716


## <font size=5> <strong>Fun Way to Test It

In [13]:
def predict_my_survival():
    print("Let's see if YOU would survive the Titanic!")
    print("-" * 40)
    
    # Ask questions
    pclass = int(input("Class (1=Rich, 2=Middle, 3=Poor): "))
    sex = input("Sex (male/female): ")
    age = float(input("Age: "))
    family = int(input("Family members with you: "))
    
    # Convert to numbers
    sex_num = 1 if sex == 'female' else 0
    alone = 1 if family == 0 else 0
    
    # Make prediction
    person = [[pclass, sex_num, age, family+1, 50, alone]]
    prob = model.predict_proba(person)[0][1]
    
    # Show result
    print("\n PREDICTION:")
    if prob > 0.7:
        print(f" {prob*100:.1f}% - You'd probably survive! (Get that lifeboat!)")
    elif prob > 0.3:
        print(f" {prob*100:.1f}% - 50-50 chance. Better run fast!")
    else:
        print(f" {prob*100:.1f}% - Tough luck. Maybe find a floating door?")

# Try it!
predict_my_survival()

Let's see if YOU would survive the Titanic!
----------------------------------------

 PREDICTION:
 97.0% - You'd probably survive! (Get that lifeboat!)


c:\Users\CVNS\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


## <font size=5> <strong>
The model achieved 79.9% accuracy by learning that:

-Women had ~74% survival rate

-Children were prioritized

-Rich people (higher fare) had better access to lifeboats

-Family size created survival patterns